In [1]:
import os
import sys
import numpy as np
import xarray as xr
import subprocess
from importlib import reload
%load_ext autoreload
%autoreload 2
sys.path.insert(0, './regional-mom6')
import regional_mom6 as rmom6


# TTide domain

In [2]:
## Choose your coordinates and the name of your experiment


yextent = [-56,-26]
xextent = [142,180]
expt_name = "testtides"

# 
daterange = ["2015-01-01 00:00:00", "2015-12-31 00:00:00"]

## Place where all your input files go
inputdir = f"{expt_name}/ttide-inputs"

## Directory where you'll run the experiment from
rundir = f"{expt_name}/ttide"

## Directory where raw downloads go before processing
tmpdir = f".temp/ttide"

## Directory where fre tools are stored
toolpath = "/usr/local/bin" ## Compiled tools needed for construction of mask tables
for i in [rundir,tmpdir,inputdir]:
    os.makedirs(str(i), exist_ok=True)

In [3]:
expt = rmom6.experiment(
        longitude_extent = xextent,
    latitude_extent = yextent,
    date_range = daterange,
    resolution = 0.05,
    number_vertical_layers = 100,
    layer_thickness_ratio = 20, 
    depth = 5500,
    mom_run_dir = rundir,
    mom_input_dir = inputdir,
    toolpath_dir = toolpath)


## Scripts for tides

The following functions have been borrowed from other repos, and modified a bit by Ashley. I've deleted unnecessary things and otherwise slimmed down the code. There's still more than is needed for our purposes! For example, they have their own boundary segment class. What we really need to do is just modify our boundary segment class to have a `setup_tides` method which works similarly to the `simple_boundary` method.

In [4]:
## The code in this file was borrowd from the North West Atlantic github repos. Not all of it was needed, and I've made some minor modifications 
#Original code: https://github.com/jsimkins2/nwa25


## The following functions and classes are from the 'boundary.py' file under /setup
import sys
from os import path
import warnings
import xarray as xarray
import xesmf
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
# ignore pandas FutureWarnings raised multiple times by xarray
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd

import numpy as np
import xarray as xr


#! Note from Ashley: The following two functions have just been copied. These can be included in our repo since the copyright info is already attached


def ap2ep(uc, vc):
    """Convert complex tidal u and v to tidal ellipse.
    Adapted from ap2ep.m for matlab
    Original copyright notice:
    %Authorship Copyright:
    %
    %    The author retains the copyright of this program, while  you are welcome
    % to use and distribute it as long as you credit the author properly and respect
    % the program name itself. Particularly, you are expected to retain the original
    % author's name in this original version or any of its modified version that
    % you might make. You are also expected not to essentially change the name of
    % the programs except for adding possible extension for your own version you
    % might create, e.g. ap2ep_xx is acceptable.  Any suggestions are welcome and
    % enjoy my program(s)!
    %
    %
    %Author Info:
    %_______________________________________________________________________
    %  Zhigang Xu, Ph.D.
    %  (pronounced as Tsi Gahng Hsu)
    %  Research Scientist
    %  Coastal Circulation
    %  Bedford Institute of Oceanography
    %  1 Challenge Dr.
    %  P.O. Box 1006                    Phone  (902) 426-2307 (o)
    %  Dartmouth, Nova Scotia           Fax    (902) 426-7827
    %  CANADA B2Y 4A2                   email xuz@dfo-mpo.gc.ca
    %_______________________________________________________________________
    %
    % Release Date: Nov. 2000, Revised on May. 2002 to adopt Foreman's northern semi
    % major axis convention.

    Args:
        uc: complex tidal u velocity
        vc: complex tidal v velocity

    Returns:
        (semi-major axis, eccentricity, inclination [radians], phase [radians])
    """
    wp = (uc + 1j * vc) / 2.0
    wm = np.conj(uc - 1j * vc) / 2.0

    Wp = np.abs(wp)
    Wm = np.abs(wm)
    THETAp = np.angle(wp)
    THETAm = np.angle(wm)

    SEMA = Wp + Wm
    SEMI = Wp - Wm
    ECC = SEMI / SEMA
    PHA = (THETAm - THETAp) / 2.0
    INC = (THETAm + THETAp) / 2.0

    return SEMA, ECC, INC, PHA


def ep2ap(SEMA, ECC, INC, PHA):
    """Convert tidal ellipse to real u and v amplitude and phase.
    Adapted from ep2ap.m for matlab.
    Original copyright notice:
    %Authorship Copyright:
    %
    %    The author of this program retains the copyright of this program, while
    % you are welcome to use and distribute this program as long as you credit
    % the author properly and respect the program name itself. Particularly,
    % you are expected to retain the original author's name in this original
    % version of the program or any of its modified version that you might make.
    % You are also expected not to essentially change the name of the programs
    % except for adding possible extension for your own version you might create,
    % e.g. app2ep_xx is acceptable.  Any suggestions are welcome and enjoy my
    % program(s)!
    %
    %
    %Author Info:
    %_______________________________________________________________________
    %  Zhigang Xu, Ph.D.
    %  (pronounced as Tsi Gahng Hsu)
    %  Research Scientist
    %  Coastal Circulation
    %  Bedford Institute of Oceanography
    %  1 Challenge Dr.
    %  P.O. Box 1006                    Phone  (902) 426-2307 (o)
    %  Dartmouth, Nova Scotia           Fax    (902) 426-7827
    %  CANADA B2Y 4A2                   email xuz@dfo-mpo.gc.ca
    %_______________________________________________________________________
    %
    %Release Date: Nov. 2000

    Args:
        SEMA: semi-major axis
        ECC: eccentricity
        INC: inclination [radians]
        PHA: phase [radians]

    Returns:
        (u amplitude, u phase [radians], v amplitude, v phase [radians])

    """
    Wp = (1 + ECC) / 2. * SEMA
    Wm = (1 - ECC) / 2. * SEMA
    THETAp = INC - PHA
    THETAm = INC + PHA

    wp = Wp * np.exp(1j * THETAp)
    wm = Wm * np.exp(1j * THETAm)

    cu = wp + np.conj(wm)
    cv = -1j * (wp - np.conj(wm))

    ua = np.abs(cu)
    va = np.abs(cv)
    up = -np.angle(cu)
    vp = -np.angle(cv)

    return ua, va, up, vp


#! Note from Ashley: This class is similar to our Segment class. We don't need most of this code, since our segment class
#! already has the attributes etc that are needed. The main thing is re-writing the regrid tidal xxx functions to work with our segmeng class

class Segment():
    """One segment of a MOM6 open boundary.

    Note that MOM6 supports segments of any length,
    but here it is assumed that the segment spans an 
    entire north, south, east, or west border. 

    Attributes:
        num (int): segment identification number following MOM6 order (1-4).
        border (str): which border of the model grid the segment represents (north, south, east, or west).
        hgrid: (xarray.Dataset) dataset from opening ocean_hgrid.nc. Contains 'x', 'y', and 'angle_dx'.
        in_degrees: (bool): is angle_dx in hgrid in units of degrees (True) or radians (False)?
        segstr (str): string identifying the segment, used in variable and file names.
        output_dir (str): location to write data for the segment, and location to store xesmf weight files.
        regrid_dir (str): location to save xesmf Regridders. Defaults to output_dir. 
        coords (xarray.Dataset): segment coordinates derived from hgrid (lon, lat, angle relative to true north).
        nx (int): Number of data points in the x direction.
        ny (int): Number of data points in the y direction.
    """

    def __init__(self, num, border, hgrid, in_degrees=True, output_dir='.', regrid_dir=None):
        self.num = num
        self.border = border
        self.hgrid = hgrid
        if in_degrees:
            self.hgrid['angle_dx'] = np.radians(self.hgrid['angle_dx'])
        self.segstr = f'segment_{self.num:03d}'
        self.output_dir = output_dir

        if regrid_dir is None:
            self.regrid_dir = self.output_dir
        else:
            self.regrid_dir = regrid_dir            

    @property
    def coords(self):
        # Rename nxp and nyp to locations
        if self.border == 'south':
            rcoord = xarray.Dataset({
                'lon': self.hgrid['x'].isel(nyp=0),
                'lat': self.hgrid['y'].isel(nyp=0),
                'angle': self.hgrid['angle_dx'].isel(nyp=0)
            })
            rcoord = rcoord.rename_dims({'nxp': 'locations'})
        elif self.border == 'north':
            rcoord = xarray.Dataset({
                'lon': self.hgrid['x'].isel(nyp=-1),
                'lat': self.hgrid['y'].isel(nyp=-1),
                'angle': self.hgrid['angle_dx'].isel(nyp=-1)
            })
            rcoord = rcoord.rename_dims({'nxp': 'locations'})
        elif self.border == 'west':
            rcoord = xarray.Dataset({
                'lon': self.hgrid['x'].isel(nxp=0),
                'lat': self.hgrid['y'].isel(nxp=0),
                'angle': self.hgrid['angle_dx'].isel(nxp=0)
            })
            rcoord = rcoord.rename_dims({'nyp': 'locations'})
        elif self.border == 'east':
            rcoord = xarray.Dataset({
                'lon': self.hgrid['x'].isel(nxp=-1),
                'lat': self.hgrid['y'].isel(nxp=-1),
                'angle': self.hgrid['angle_dx'].isel(nxp=-1)
            })
            rcoord = rcoord.rename_dims({'nyp': 'locations'})

        # Make lat and lon coordinates
        rcoord = rcoord.assign_coords(
            lat=rcoord['lat'],
            lon=rcoord['lon']
        )

        return rcoord

    @property
    def nx(self):
        """Number of data points in the x-direction"""
        if self.border in ['south', 'north']:
            return len(self.coords['lon'])
        elif self.border in ['west', 'east']:
            return 1
    
    @property
    def ny(self):
        """Number of data points in the y-direction"""
        if self.border in ['south', 'north']:
            return 1
        elif self.border in ['west', 'east']:
            return len(self.coords['lat'])
    
    #! Note from Ashley. This encoding information should be included. Could just implement these steps within our 'setup_tides' method rather than copying a whole function that's only used twice. 
    def to_netcdf(self, ds, varnames, suffix=None):
        """Write data for the segment to file.

        Args:
            ds (xarray.Dataset): Segment dataset.
            varnames (str): Name to give the file (e.g. 'temp', 'salt'). 
            suffix (str, optional): Optional suffix to append to the filename (before .nc). Defaults to None.
        """
        for v in ds:
            ds[v].encoding['_FillValue']= 1.0e20
        fname = f'{varnames}_{self.num:03d}_{suffix}.nc' if suffix is not None else f'{varnames}_{self.num:03d}.nc'
        # Set format and attributes for coordinates, including time if it does not already have calendar attribute
        # (may change this to detect whether time is a time type or a float).
        # Need to include the fillvalue or it will be back to nan
        encoding = {
            'time': dict(_FillValue=1.0e20),
            f'lon_{self.segstr}': dict(dtype='float64', _FillValue=1.0e20),
            f'lat_{self.segstr}': dict(dtype='float64', _FillValue=1.0e20)
        }
        if 'calendar' not in ds['time'].attrs and 'modulo' not in ds['time'].attrs:
            encoding.update({'time': dict(dtype='float64', calendar='gregorian', _FillValue=1.0e20)})
        ds.to_netcdf(
            path.join(self.output_dir, fname),
            format='NETCDF3_64BIT',
            engine='netcdf4',
            encoding=encoding,
            unlimited_dims='time'
        )


        
   
    def regrid_tidal_elevation(
                self, resource, imsource, time, 
                method='nearest_s2d', periodic=False, write=True, 
                flood=False, xdim='nx', ydim='ny', **kwargs):
        """Regrid tidal elevation onto segment and (optionally) write to file.
        It is assumed that real (resource) and imaginary (imsource) components of the 
        constituents have the same coordinates.

        Args:
            resource (xarray.DataArray): Real component of tidal elevation on source grid.
            imsource (xarray.DataArray): Imaginary component of tidal elevation on source grid.
            time: Time to add to dataset. Pass a length-1 array to keep it constant in time.
            method (str, optional): Method recognized by xesmf to use to regrid. Defaults to 'nearest_s2d'.
            periodic (bool, optional): Whether the source grid is periodic (passed to xesmf). Defaults to False.
            write (bool, optional): After regridding, write the results to file. Defaults to True.
            flood (bool, optional): As the first step of regridding, horizontally flood the source data. Defaults to False.
            xdim (str, optional): Name of the horizontal x dimension, needed if flooding. Defaults to 'nx'.
            ydim (str, optional): Name of the horizontal y dimension, needed if flooding. Defaults to 'ny'.
            **kwargs: additional keyword arguments passed to Segment.to_netcdf().

        Returns:
            xarray.Dataset: Dataset of regridded boundary data.
        """

        # Horizontally interpolate elevation components
        regrid = xesmf.Regridder(
            resource,
            self.coords,
            method=method,
            locstream_out=True,
            periodic=periodic,
            filename=path.join(self.regrid_dir, f'regrid_{self.segstr}_tidal_elev.nc'),
            reuse_weights=False
        )
        redest = regrid(resource)
        imdest = regrid(imsource)

        # Fill missing data.
        # Need to do this first because complex would get converted to real
        redest = redest.ffill(dim="locations", limit=None)['hRe']
        imdest = imdest.ffill(dim="locations", limit=None)['hIm']

        # Convert complex
        cplex = redest + 1j * imdest

        # Convert to real amplitude and phase.
        ds_ap = xarray.Dataset({
            f'zamp_{self.segstr}': np.abs(cplex)
        })
        # np.angle doesn't return dataarray
        ds_ap[f'zphase_{self.segstr}'] =  (('constituent', 'locations'), -1 * np.angle(cplex))  # radians

        # Add time coordinate and transpose so that time is first,
        # so that it can be the unlimited dimension
        ds_ap, _ = xarray.broadcast(ds_ap, time)
        ds_ap = ds_ap.transpose('time', 'constituent', 'locations')

        ds_ap = self.expand_dims(ds_ap)
        ds_ap = self.rename_dims(ds_ap)

        if write:
            self.to_netcdf(ds_ap, 'tz', **kwargs)
            
        return ds_ap

    def regrid_tidal_velocity(
            self, uresource, uimsource, vresource, vimsource, time, 
            method='nearest_s2d', periodic=False, write=True, 
            flood=False, xdim='nx', ydim='ny', **kwargs):
        """Regrid tidal velocity onto segment and (optionally) write to file.
        It is assumed that real and imaginary components of the 
        individual u or v velocities have the same coordinates, 
        but the u and v components may have separate coordinates
        [although currently they must have the same names if flooding].

        Args:
            uresource (xarray.DataArray): Real component of tidal u velocity on source grid.
            uimsource (xarray.DataArray): Imaginary component of tidal u velocity on source grid.
            vresource (xarray.DataArray): Real component of tidal v velocity on source grid.
            vimsource (xarray.DataArray): Imaginary component of tidal v velocity on source grid.
            time: Time to add to dataset. Pass a length-1 array to keep it constant in time.
            method (str, optional): Method recognized by xesmf to use to regrid. Defaults to 'nearest_s2d'.
            periodic (bool, optional): Whether the source grid is periodic (passed to xesmf). Defaults to False.
            write (bool, optional): After regridding, write the results to file. Defaults to True.
            flood (bool, optional): As the first step of regridding, horizontally flood the source data. Defaults to False.
            xdim (str, optional): Name of the horizontal x dimension, needed if flooding. Defaults to 'nx'.
            ydim (str, optional): Name of the horizontal y dimension, needed if flooding. Defaults to 'ny'.
            **kwargs: additional keyword arguments passed to Segment.to_netcdf().

        Returns:
            xarray.Dataset: Dataset of regridded boundary data.
        """
        regrid_u = xesmf.Regridder(
            uresource,
            self.coords,
            method=method,
            locstream_out=True,
            periodic=periodic,
            # filename=path.join(self.regrid_dir, f'regrid_{self.segstr}_tidal_u.nc'),
            reuse_weights=False
        )

        regrid_v = xesmf.Regridder(
            vresource,
            self.coords,
            method=method,
            locstream_out=True,
            periodic=periodic,
            # filename=path.join(
            #     self.regrid_dir, f'regrid_{self.segstr}_tidal_v.nc'),
            reuse_weights=False
        )

        # Interpolate each real and imaginary parts to segment.
        uredest = regrid_u(uresource)['uRe']
        uimdest = regrid_u(uimsource)['uIm']
        vredest = regrid_v(vresource)['vRe']
        vimdest = regrid_v(vimsource)['vIm']

        # Fill missing data.
        # Need to do this first because complex would get converted to real
        uredest = uredest.ffill(dim="locations", limit=None)
        uimdest = uimdest.ffill(dim="locations", limit=None)
        vredest = vredest.ffill(dim="locations", limit=None)
        vimdest = vimdest.ffill(dim="locations", limit=None)

        # Convert to complex, remaining separate for u and v.
        ucplex = uredest + 1j * uimdest
        vcplex = vredest + 1j * vimdest

        # Convert complex u and v to ellipse,
        # rotate ellipse from earth-relative to model-relative,
        # and convert ellipse back to amplitude and phase.
        # There is probably a complicated trig identity for this? But
        # this works too. 
        # if self.border in ['south', 'north']:
        #     angle = self.coords['angle'].rename({'nxp': 'locations'})
        # elif self.border in ['west', 'east']:
        #     angle = self.coords['angle'].rename({'nyp': 'locations'})
        SEMA, ECC, INC, PHA = ap2ep(ucplex, vcplex)



        # Rotate to the model grid by adjusting the inclination.
        # Requries that angle is in radians.
        # INC is np array but angle is xarray
        # INC -= angle.data[np.newaxis, :]
        ua, va, up, vp = ep2ap(SEMA, ECC, INC, PHA)

        ds_ap = xarray.Dataset({
            f'uamp_{self.segstr}': ua,
            f'vamp_{self.segstr}': va
        })
        # up, vp aren't dataarrays
        ds_ap[f'uphase_{self.segstr}'] =  (('constituent', 'locations'), up)  # radians
        ds_ap[f'vphase_{self.segstr}'] =  (('constituent', 'locations'), vp)  # radians

        ds_ap, _ = xarray.broadcast(ds_ap, time)

        # Need to transpose so that time is first,
        # so that it can be the unlimited dimension
        ds_ap = ds_ap.transpose('time', 'constituent', 'locations')

        # Some things may have become missing during the transformation
        ds_ap = ds_ap.ffill(dim="locations", limit=None)

        ds_ap = self.expand_dims(ds_ap)
        ds_ap = self.rename_dims(ds_ap)

        if write:
            self.to_netcdf(ds_ap, 'tu', **kwargs)
            
        return ds_ap



## The following is from the /setup/tides folder. This function reads in your tpxo file and cuts out the parts along the boundary.

# From https://github.com/jsimkins2/nwa25/tree/main/setup/tides

def write_tpxo(constituents, tpxo_dir, segments, horizontal_subset,filename):
    tpxo_h = (
        xarray.open_dataset(path.join(tpxo_dir, f'h_{filename}'))
        .rename({'lon_z': 'lon', 'lat_z': 'lat', 'nc': 'constituent'})
        .isel(constituent=constituents, **horizontal_subset)
    )
    h = tpxo_h['ha'] * np.exp(-1j * np.radians(tpxo_h['hp']))
    tpxo_h['hRe'] = np.real(h)
    tpxo_h['hIm'] = np.imag(h)
    tpxo_u = (
        xarray.open_dataset(path.join(tpxo_dir, f'u_{filename}'))
        .rename({'lon_u': 'lon', 'lat_u': 'lat', 'nc': 'constituent'})
        .isel(constituent=constituents, **horizontal_subset)
    )
    tpxo_u['ua'] *= 0.01  # convert to m/s
    u = tpxo_u['ua'] * np.exp(-1j * np.radians(tpxo_u['up']))
    tpxo_u['uRe'] = np.real(u)
    tpxo_u['uIm'] = np.imag(u)
    tpxo_v = (
        xarray.open_dataset(path.join(tpxo_dir, f'u_{filename}'))
        .rename({'lon_v': 'lon', 'lat_v': 'lat', 'nc': 'constituent'})
        .isel(constituent=constituents, **horizontal_subset)
    )
    tpxo_v['va'] *= 0.01  # convert to m/s
    v = tpxo_v['va'] * np.exp(-1j * np.radians(tpxo_v['vp']))
    tpxo_v['vRe'] = np.real(v)
    tpxo_v['vIm'] = np.imag(v)
    # Tidal amplitudes are currently constant over time.
    # Seem to need a time dimension to have it read by MOM.
    # But also, this would later allow nodal modulation
    # or other long-term variations to be added.
    # the date should begin 1 month before first day of simulation
    times = xarray.DataArray(
        pd.date_range('2009-12-01', periods=1),
        dims=['time']
    )
    for seg in segments:
        print(seg)
        seg.regrid_tidal_elevation(
            tpxo_h[['lon', 'lat', 'hRe']],
            tpxo_h[['lon', 'lat', 'hIm']],
            times,
            flood=False
        )
        seg.regrid_tidal_velocity(
            tpxo_u[['lon', 'lat', 'uRe']],
            tpxo_u[['lon', 'lat', 'uIm']],
            tpxo_v[['lon', 'lat', 'vRe']],
            tpxo_v[['lon', 'lat', 'vIm']],
            times,
            flood=False
        )



## Add Tides to an Existing Run
Assuming you have run the above, this will read in your existing `hgrid` to set up tides at boundaries 

In [ ]:

constituents = [0] # This selects only M2. Set to range(0,10) for all tidal constituents

# Hardcoded for my ttide domain!!! Choose indices that work for you.

horizontal_subset = dict(ny=slice(200,380), nx=slice(850,1100)) # This is hardcoded! the TPXO file only contains nx and ny not latlon
#todo: subset based on lat lon of experiment

# Path to tpxo9.v1 data
tpxo_dir = 'inputs_rm6/tidal_data'

# Save output here
output_dir = inputdir + "/forcing"

# Setup NWA boundaries
hgrid = xr.open_dataset(inputdir + '/hgrid.nc')
# segments taken from nwa25_obc.py
segments = [
    Segment(1, 'south', hgrid, output_dir=output_dir),
    Segment(2, 'north', hgrid, output_dir=output_dir),
    Segment(3, 'west', hgrid, output_dir=output_dir),
    Segment(4, 'east', hgrid, output_dir=output_dir)
]
write_tpxo(constituents, tpxo_dir, segments, horizontal_subset,"tpxo9.v1.nc")









In [19]:
tpxo_dir = '/Users/manishrv/Documents/nwa12_v0/regional_mom_workflows/rm6/inputs_rm6/tidal_data'
expt.setup_tides(tpxo_dir, "tpxo9.v1.nc", horizontal_subset)

Processing tides at south boundary...Done.
Processing tides at north boundary...Done.
Processing tides at west boundary...Done.
Processing tides at east boundary...Done.
